In [1]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from pypdf import PdfReader

In [2]:
import re

reader = PdfReader("data.pdf")
text = ""
for page in reader.pages:
    text += page.extract_text()

# Clean the text
text = re.sub(r'[▪•▸►‣]', '', text)        # remove bullet symbols
text = re.sub(r'\s+', ' ', text)             # collapse extra whitespace
text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # remove non-ASCII characters
text = text.strip()

print("Cleaned text length:", len(text))
print("Sample:", text[:300])  # check it looks clean

Cleaned text length: 10901
Sample: PREPARING FOR AN INTERVIEW BEFORE THE INTERVIEW Research the company and position description by using their company website, CEO resources, LinkedIn, and networking with contacts and employees. Be prepared to give specific examples from your experience, education, or skills that are relevant to the


In [3]:
def chunk_text(text, chunk_size = 600, overlap = 100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

chunks = chunk_text(text)

print("Total chunks :", len(chunks))

Total chunks : 22


In [4]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(chunks)

In [5]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS Index Ready")

FAISS Index Ready


In [6]:
def retrieve(query, k=3):
    query_vector = embedding_model.encode([query])
    distances, indices = index.search(query_vector, k)
    results = [chunks[i] for i in indices[0]]
    return results

In [7]:
generator = pipeline("text2text-generation", model = 'google/flan-t5-large')

Device set to use cpu


In [15]:
import re

def rag_answer(query):
    retrieved_chunks = retrieve(query, k=2)
    
    # Join and clean — remove incomplete sentence at the start
    context = " ".join(retrieved_chunks)[:800]
    
    # Remove any fragment before the first capital letter
    context = re.sub(r'^[^A-Z]+', '', context)

    prompt = f"Based on this text: {context} Answer this in detail: {query}"

    result = generator(
        prompt,
        max_new_tokens=250,
        do_sample=False,
        truncation=True
    )

    raw = result[0]['generated_text'].strip()

    if not raw or len(raw) < 5:
        # Fallback: search directly in all chunks for relevant sentences
        all_text = " ".join(chunks)
        all_text = re.sub(r'^[^A-Z]+', '', all_text)
        prompt2 = f"Based on this text: {all_text[:800]} Answer this: {query}"
        result2 = generator(prompt2, max_new_tokens=250, do_sample=False, truncation=True)
        raw = result2[0]['generated_text'].strip()

    if not raw or len(raw) < 5:
        return "Sorry, I couldn't find an answer in the document."

    sentences = re.split(r'(?<=[.!?])\s+', raw)
    
    # Filter out broken fragments (must start with capital letter)
    bullets = [f"• {s.strip()}" for s in sentences 
               if len(s.strip()) > 15 and s.strip()[0].isupper()]
    
    return "\n".join(bullets[:7]) if bullets else raw


# Dynamic chat loop
print("RAG Chatbot ready! Type 'quit' to exit.\n")

while True:
    query = input("Ask a question about the document: ").strip()
    if query.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break
    if not query:
        continue
    print("\nAnswer:")
    print(rag_answer(query))
    print("-" * 50)

RAG Chatbot ready! Type 'quit' to exit.



Ask a question about the document:  Should I ask about salary in the first interview?



Answer:
• Avoid asking about salary and benefits during the first interview.
• Ask at least two of your pre-prepared, well thought out questions to determine if this organization and job is right place for you.
• Main a good balance between getting your points across and dominating the conversation.
--------------------------------------------------


Ask a question about the document:  I feel nervous in interviews, what should I do?



Answer:
• You need to be fully engaged in this conversation, so turn off your cell phone and do not check it.
• Do not chew gum, eat or drink (unless offered to you).
• Be honest and sincere.
• Just be yourself!
• AFTER THE INTERVIEW Send a thank you letter within two days to the people who interviewed you.
• Reiterate your interest and some key points that were discussed during the interview.
• You can also use thank you notes to expand on a po ice, practice!
--------------------------------------------------


Ask a question about the document:  quit


Goodbye!
